### Install new libraries

In [ ]:
#!pip install ddgs trafilatura -q # -q without any logs
#!pip install openai-agents

### Setup imports

In [7]:
import os
from dotenv import load_dotenv
from pprint import pprint
import json
from ddgs import DDGS
import trafilatura
from IPython.display import display, Markdown

from agents import Agent, Runner, function_tool

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API Key is Missing")

MODEL="gpt-4o-mini"

### Step: Function tools

In [8]:
@function_tool
def search_web(query: str):
    """ Search the web using DuckDuckGo browser. Return 3 results."""
    ddgs = DDGS()
    results = ddgs.text(query, max_results=10)
    print(f"\u2705 got results")
    return json.dumps(results, indent=2)

In [13]:
@function_tool
def fetch_url(url: str):
    """Fetch the content of a URL using trafilatura"""
    downloaded = trafilatura.fetch_url(url)
    if downloaded:
        text = trafilatura.extract(downloaded)
        if text:
            print(f" \u2705 Got text: {len(text)} chars")
            return text
    print(f"\u274c Failed to fetch or extract test fron {url}.")
    return f"Could not extract the text from {url}. try a different source"

### Step 2: The System Prompt
#### This tells the LLM who it is and how to behave. 
#### The key things:
- what its job is
- What tool it has
- what process to follow
- what output formay to produce

In [10]:
RESEARCH_AGENT_PROMPT = """You are a research specialist. Your job is to research a given topic
and produce a comprehensive research brief.

You have access to two tools:
- search_web: Search the web for information
- fetch_url: Fetch and read the full content of a web page

Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results — which sources look most relevant and why?
3. Fetch the full content of the 2-3 best URLs
4. Reflect on what you have gathered. Do you have enough? Are there gaps?
5. If there are gaps, search again with a different query
6. When you have enough information from at least 6 different sources, synthesize into a research brief

You MUST gather information from at least 6 distinct sources before delivering your brief. 
If you have fewer than 4 sources, keep searching.

Your research brief MUST include:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

Until you are ready, just keep working — search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step.
Not every response needs a tool call — sometimes just thinking through what you have is the right move."""

### Step 3: Define the Agent

In [14]:
research_agent = Agent(
    name = "Research Agent",
    instructions = RESEARCH_AGENT_PROMPT,
    model = MODEL,
    tools = [search_web, fetch_url]
)

### Run Agent

In [15]:
result = await Runner.run(
    research_agent,
    input = "Reearch the following topic and produce a comprehensive research brief: how AI is used in healthcare",
    max_turns=30
)

✅ got results
❌ Failed to fetch or extract test fron https://www.iso.org/artificial-intelligence/what-is-ai.
 ✅ Got text: 3010 chars
 ✅ Got text: 1747 chars
 ✅ Got text: 4184 chars
 ✅ Got text: 2378 chars
❌ Failed to fetch or extract test fron https://www.ahrq.gov/ncepcr/tools/healthcare-ai.html.
 ✅ Got text: 131 chars
❌ Failed to fetch or extract test fron https://www.healthaffairs.org/doi/10.1377/hlthaff.2020.01215.
 ✅ Got text: 131 chars
✅ got results
 ✅ Got text: 5582 chars


In [16]:
print(f"Agent: {result.last_agent.name}")
print(f"-----")
display(Markdown(result.final_output))

Agent: Research Agent
-----


### Research Brief: The Use of AI in Healthcare

#### Key Facts and Statistics
1. **Adoption Rates**: As of 2024, **66%** of physicians reported using AI in their practices, a significant increase from **38%** in 2023.
2. **Technological Integration**: AI is employed for various healthcare tasks, including documentation of billing codes (21%), creation of discharge instructions (20%), and assistive diagnosis (12%).
3. **Investment in Mental Health AI**: Startups like **Aiberry**, which analyses voice and conversation to screen for mental health issues, raised **$8 million** in funding, demonstrating investment in this innovative space.
4. **Preventive Health Technologies**: Companies like **Neko Health**, backed by Spotify founder Daniel Ek, utilize AI for non-invasive body scans to detect health abnormalities, emphasizing preventive care.

#### Main Themes and Arguments
1. **Transformative Potential**: AI is positioned as a game-changer for healthcare, with applications ranging from patient diagnostics to administrative tasks. The integration of **Generative AI** and **Natural Language Processing** is expected to enhance patient care and streamline operations within healthcare facilities.
   
2. **Mental Health Applications**: A significant trend is the use of AI for mental health screening. Tools like **Aiberry** offer alternatives to traditional screening methods by using conversational analysis to assess mental well-being quickly and effectively.

3. **Preventive Care Focus**: Startups such as **Neko Health** illustrate a shift towards preventive care. Their AI-driven full-body scanners detect various health indicators, highlighting a proactive approach to health management.

4. **Skepticism and Regulation**: Despite the growing adoption of AI, there remains skepticism among healthcare professionals. Concerns about data privacy, AI integration with existing systems, and the accuracy of AI-driven recommendations necessitate regulatory oversight and enhanced training for practitioners.

5. **AI as an Assistant**: The consensus among many healthcare providers is that AI should serve as an assistive tool, reducing burdens and improving diagnostics rather than replacing human judgment. There’s recognition of AI’s potential benefits, yet caution remains regarding trust in the technology's integrity and its implementation.

#### Notable Data Points
- **Utilization Growth**: Physicians using AI for documentation jumped from 13% to 21% in a year.
- **Key Opportunities**: Over half of the surveyed physicians (57%) identified the opportunity for AI to alleviate administrative tasks as a primary advantage.
- **Privacy and Trust**: Nearly half (47%) of surveyed physicians indicated the need for increased oversight to bolster trust in AI adoption.

#### Conclusion
AI applications in healthcare are expanding rapidly, offering promising advancements in diagnostics, patient care, and operational efficiencies. However, the sector faces challenges regarding integration, trust, and effective application. As the technology matures, a balanced approach involving regulatory frameworks and professional training will be crucial in maximizing the benefits of AI in healthcare. 

#### Sources
- Aiberry (GeekWire) - [Link](https://www.geekwire.com/2023/startup-developing-an-ai-powered-mental-health-screening-tool-lands-8m/)
- Thumbay Institute (Thumbay Moideen) - [Link](https://thumbaymoideen.com/thumbay-institute-for-ai-in-healthcare-hosts-transformative-workshop-on-generative-ai-and-natural-language-processing-in-healthcare-applications/)
- AI and Physician Sentiment (AMA) - [Link](https://www.ama-assn.org/practice-management/digital-health/2-3-physicians-are-using-health-ai-78-2023)
- Neko Health (The Verge) - [Link](https://www.theverge.com/2023/2/4/23585948/spotify-founder-daniel-ek-ai-body-health-scanner)
- Large Language Models in Health (ENTtoday) - [Link](https://www.enttoday.org/article/large-language-model-ai-technology-is-being-used-in-some-healthcare-applications-but-is-it-any-good/3/) 

This brief synthesizes recent developments in AI healthcare applications and aims to inform stakeholders about the current landscape, opportunities, and challenges within the field.